In [1]:
from nn_framework import *
import numpy as np
import pandas as pd

In [2]:
def create_sample_csv(filename="train.csv", n_samples=300):
    X = np.random.randn(n_samples, 3)

    y = np.zeros(n_samples, dtype=int)
    sums = np.sum(X, axis=1)
    y[sums < -1] = 0
    y[(sums >= -1) & (sums <= 1)] = 1
    y[sums > 1] = 2

    df = pd.DataFrame(X, columns=['f1', 'f2', 'f3'])

    targets = pd.get_dummies(y, prefix='target')
    df = pd.concat([df, targets], axis=1)

    df.to_csv(filename, index=False)
    print(f"Файл {filename} успешно создан!")
    print(df.head())

In [3]:
create_sample_csv()

Файл train.csv успешно создан!
         f1        f2        f3  target_0  target_1  target_2
0  0.481334  1.386580  0.534932     False     False      True
1  0.584634 -1.219077  1.769475     False     False      True
2  1.735004 -1.497100  1.134830     False     False      True
3 -1.310249  0.287831 -0.417515      True     False     False
4  0.298599  0.623007  0.523956     False     False      True


In [11]:
target_columns = ['target_0', 'target_1', 'target_2']
dataset = Dataset.from_csv("train.csv", target_cols=target_columns)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

net = Sequential(
    Linear(3, 16),
    ReLU(),
    Linear(16, 3),
    Softmax()
)

optimizer = Adam(net.parameters(), lr=0.03)
optimizer = GradientClipping(optimizer, clip_value=1.0)

model = Model(
    network=net,
    loss_fn=CrossEntropy(),
    optimizer=optimizer,
    metrics=[Accuracy(), Precision(), Recall()]
)

print("Начинаем обучение...")
model.fit(loader, epochs=20)

test_x = [[0.5, 0.8, 0.1]]
prediction = model.predict(test_x)
print(f"\nПредсказание для {test_x}:")
print(f"Вероятности классов: {prediction}")
print(f"Выбранный класс: {np.argmax(prediction)}")

model.evaluate(loader)

Начинаем обучение...
Epoch [1/20] - loss: 0.8636 - accuracy: 0.8133 - precision: 0.8571 - recall: 0.6800 - 0.01s
Epoch [2/20] - loss: 0.3847 - accuracy: 0.9133 - precision: 0.9164 - recall: 0.9133 - 0.01s
Epoch [3/20] - loss: 0.2255 - accuracy: 0.9433 - precision: 0.9431 - recall: 0.9400 - 0.01s
Epoch [4/20] - loss: 0.1810 - accuracy: 0.9667 - precision: 0.9667 - recall: 0.9667 - 0.01s
Epoch [5/20] - loss: 0.1372 - accuracy: 0.9767 - precision: 0.9767 - recall: 0.9767 - 0.01s
Epoch [6/20] - loss: 0.1262 - accuracy: 0.9667 - precision: 0.9667 - recall: 0.9667 - 0.01s
Epoch [7/20] - loss: 0.1492 - accuracy: 0.9567 - precision: 0.9567 - recall: 0.9567 - 0.01s
Epoch [8/20] - loss: 0.1055 - accuracy: 0.9800 - precision: 0.9800 - recall: 0.9800 - 0.01s
Epoch [9/20] - loss: 0.0851 - accuracy: 0.9933 - precision: 0.9933 - recall: 0.9933 - 0.01s
Epoch [10/20] - loss: 0.0850 - accuracy: 0.9700 - precision: 0.9700 - recall: 0.9700 - 0.01s
Epoch [11/20] - loss: 0.0890 - accuracy: 0.9900 - precisio

{'accuracy': np.float64(0.9833333333333333),
 'precision': np.float64(0.9833333333333333),
 'recall': np.float64(0.9833333333333333)}

In [2]:
from nn_framework.evolution import *
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

In [3]:
def prepare_data(n_samples=2000, n_features=20, n_classes=4):
    x, y_indices = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=15,
        n_classes=n_classes,
        random_state=42
    )

    scaler = StandardScaler()
    x = scaler.fit_transform(x)

    y = np.zeros((n_samples, n_classes))
    y[np.arange(n_samples), y_indices] = 1

    split = int(n_samples * 0.8)

    train_ds = Dataset(x[:split], y[:split])
    val_ds = Dataset(x[split:], y[split:])

    return train_ds, val_ds

In [4]:
train_ds, val_ds = prepare_data()
train_loader = DataLoader(train_ds, batch_size=32)
val_loader = DataLoader(val_ds, batch_size=32)
space = SearchSpace(
    activations=[ReLU, LeakyReLU, Tanh],
    hidden_sizes=[16, 32, 64, 128],
    lr_rates=[0.05, 0.01, 0.005, 0.001],
    optimizers=[Adam, SGD, MomentumSGD],
    min_layers=1,
    max_layers=4
)

cfg = EvolutionConfig(
    pop_size=8,
    generations=12,
    eval_epochs=3,
    survival_rate=0.5,
    overfit_penalty=0.6
)

engine = EvolutionaryEngine(
    input_dim=20,
    output_dim=4,
    loss_fn=CrossEntropy(),
    metrics=[Accuracy()],
    search_space=space,
    config=cfg,
    output_activation=Softmax
)

best_dna = engine.run(train_loader, val_loader)

print("\nTraining Best Model...")
best_model = best_dna.build_model(CrossEntropy(), [Accuracy()], Softmax)
best_model.fit(train_loader, epochs=15, val_loader=val_loader)


--- Generation 1/12 ---
Epoch [1/3] - loss: 1.3466 - accuracy: 0.3919 - 0.03s
Epoch [2/3] - loss: 1.1633 - accuracy: 0.6206 - 0.03s
Epoch [3/3] - loss: 0.8884 - accuracy: 0.7262 - 0.02s
Epoch [1/3] - loss: 1.5214 - accuracy: 0.2100 - 0.01s
Epoch [2/3] - loss: 1.4925 - accuracy: 0.2175 - 0.01s
Epoch [3/3] - loss: 1.4664 - accuracy: 0.2356 - 0.01s
Epoch [1/3] - loss: 1.3819 - accuracy: 0.2781 - 0.03s
Epoch [2/3] - loss: 1.3799 - accuracy: 0.2794 - 0.03s
Epoch [3/3] - loss: 1.3779 - accuracy: 0.2888 - 0.04s
Epoch [1/3] - loss: 1.3654 - accuracy: 0.4387 - 0.08s
Epoch [2/3] - loss: 1.3297 - accuracy: 0.5356 - 0.14s
Epoch [3/3] - loss: 1.2867 - accuracy: 0.5681 - 0.08s
Epoch [1/3] - loss: 1.3860 - accuracy: 0.2681 - 0.02s
Epoch [2/3] - loss: 1.3648 - accuracy: 0.4225 - 0.02s
Epoch [3/3] - loss: 1.3159 - accuracy: 0.4819 - 0.02s
Epoch [1/3] - loss: 1.3874 - accuracy: 0.2600 - 0.08s
Epoch [2/3] - loss: 1.3801 - accuracy: 0.2725 - 0.08s
Epoch [3/3] - loss: 1.3730 - accuracy: 0.2850 - 0.07s
Epo

In [8]:
print(best_model.evaluate(val_loader))
best_model.network.layers

{'accuracy': np.float64(0.875)}


(<nn_framework.layers.Linear at 0x2056d71fb90>,
 <nn_framework.activations.Softmax at 0x2056a4496d0>)